### Part 1: Setup

In [2]:
# Install Unsloth (handles LoRA, quantization, fast training)
# !pip install -q unsloth
# !pip install -q datasets evaluate rouge_score

# Supress noisy deprication warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*use_return_dict.")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")


In [4]:
import torch, json, random, math, re, os
from unsloth import FastLanguageModel
from pathlib import Path

print(f"PyTorch:  {torch.__version__}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch:  2.10.0+cu128
GPU:      NVIDIA GeForce RTX 3070
VRAM:     8.6 GB


### Load Base Model

In [ ]:
BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
MAX_SEQ_LENGTH = 512
SEED = 42

# Load the base model (no fine-tuning yet)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Set up for inference
FastLanguageModel.for_inference(model)

print(f"\n[OK] Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[OK] Model loaded: HuggingFaceTB/SmolLM-135M
Parameters: 81430848


In [ ]:
# Helper function to generate text
def generate_text(model, tokenizer, promt, max_new_tokens=100):
  """Generate a completion fo a given prompt"""
  inputs = tokenizer(
      promt,
      return_tensors="pt",
      truncation=True,
      max_length=256
  ).to(model.device)

  with torch.no_grad():
    outputs = model.generate(
      inputs=inputs.input_ids,
      attention_mask=inputs.attention_mask,
      max_new_tokens=max_new_tokens,
      use_cache=True,
      do_sample=True,
      repetition_penalty=1.2,
      temperature=0.7,
      top_p=0.9,
    )

  # Only decode the new tokens (not the prompt)
  new_tokens = outputs[0, inputs.input_ids.shape[1]:]
  return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
# Test the base model on financial prompts
financial_prompts = [
  "The company reported total revenue of",
  "Risk factors include exposure to fluctuations in",
  "Diluted earnings per share for the fiscal year ended",
  "The Board of Directors declared a quarterly dividend of",
  "Our long-term debt obligations consist primarily of",
]

print("=" * 60)
print("BASE MODEL -- Financial Text Generation")
print("=" * 60)
for prompt in financial_prompts:
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
  print("-" * 60)
  print(f"\nPrompt: {prompt}")
  print(f"Completion: {completion}")
  print("-" * 60)

Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL -- Financial Text Generation


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------

Prompt: The company reported total revenue of
Completion:  $150,746.
Kraft Foods is also one the largest manufacturers in the U.S., with 23 manufacturing plants across America as well as international locations such as France and Italy (9). The Kraft Company was founded by John W. Kraft Jr who
------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------

Prompt: Risk factors include exposure to fluctuations in
Completion:  stress levels, chronic pain and other health issues.
The study also shows that treatment of patients with spinal cord injury is effective at reducing anxiety symptoms while increasing happiness during the early phase after diagnosis (30). This results from decreased brain activity leading up to post-injury recovery rather than being caused
------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------

Prompt: Diluted earnings per share for the fiscal year ended
Completion:  December 2016 was $438,579 as compared to a median of just over $76,500. The average income in 2017 reached an impressive figure—$79,783 (according to data from Forbes)—while
------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------

Prompt: The Board of Directors declared a quarterly dividend of
Completion:  500,000 dollars. The shareholder has to pay the company at least twice more than once for each year in total dividends (6% from monthly and annual) – i.e., if there was an increase on top-level management’s salary as well with regard to
------------------------------------------------------------
------------------------------------------------------------

Prompt: Our long-term debt obligations consist primarily of
Completion:  interest payments, which represent around 90% or nearly all the remainder. Interest rates vary depending on factors such as creditworthiness and chosen financing providers (e.g., banks).
The majority of corporate creditors owe more than $2 trillion – that's roughly equal to almost two US households!
------------------------------------------------------------


Observation: The base model generates generic text. It has no idea what financial language sounds like. It produces grammatical English, but i won't sound like a 10-K filing.

In [ ]:
# measures how surprised model is by comparing generated token with actual next token
def compute_preplixity(model, tokenizer, texts, max_length=512):
  """
  Preplexty = e^(avg cross-entropy loss)
  lower = model is lsess surprised by the text = better fit

  This is the metric for CPT. If preplexity drops on domain text
  after trainning, the model learned the domain.
  """
  total_loss = 0
  total_tokens = 0

  model.eval()
  for text in texts:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    with torch.no_grad():
      outputs = model(**inputs, labels=inputs.input_ids)

    num_tokens = inputs.input_ids.shape[1]
    total_loss += outputs.loss.item() * num_tokens
    total_tokens += num_tokens

  avg_loss = total_loss / total_tokens
  return math.exp(avg_loss)

In [ ]:
# Quick preplexity check on sample financial text
sample_financial = [
  "The Company recognized impairment charges of $142 million related to goodwill associated with the North America reporting unit during the fiscal year ended March 31, 2024.",
  "Net cash provided by operating activities was $3.8 billion for the year ended December 31, 2023, compared to $4.1 billion for the year ended December 31, 2022.",
  "Total stockholders equity decreased to $12.4 billion as of September 30, 2023, primarily due to share repurchases of $2.1 billion and dividend payments of $800 million.",
]

base_ppl = compute_preplixity(model, tokenizer, sample_financial)

# Higher = more surprised = doesn't know domain well
print(f"Base Model preplexity on finiancial text: {base_ppl:.1f}")


Base Model preplexity on finiancial text: 11.3


### Prepare the Data

In [ ]:
from datasets import load_dataset

print("[...] Loading SEC 10-K filings from HuggingFace...")
print("This is the PleIAs/SEC dataset -- full annual reports 1993-2024.")

# Stream to avoid downloadingn 100GB+
sec_dataset = load_dataset(
    "PleIAs/SEC",
    split="train",
    streaming=True,
)

# Collect a subset of filings
NUM_TRAIN = 80
NUM_VAL = 20
all_texts = []

print(f"[...] Collecting {NUM_TRAIN + NUM_VAL} filings...")
for i, example in enumerate(sec_dataset):
  text = example.get("text", "")
  # Filter: only keep substantial filings (>1000 words)
  if text and len(text.split()) > 1000:
    all_texts.append(text)
  if len(all_texts) >= NUM_TRAIN + NUM_VAL:
    break
  if i % 500 == 0 and i > 0:
    print(f"  Scanned {i} entries, collected {len(all_texts)} so far...")

print(f"\n[OK] Collected {len(all_texts)} filings.")

[...] Loading SEC 10-K filings from HuggingFace...
This is the PleIAs/SEC dataset -- full annual reports 1993-2024.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

[...] Collecting 100 filings...

[OK] Collected 100 filings.


In [ ]:
# Let's look at what a 10-K filing looks like
sample = all_texts[0]
words = sample.split()
print(f"Sample filing length: {len(words)} words")
print(f"\nFirst 200 words:")
print(" ".join(words[:200]))
print("\n...")
print(f"\nLast 100 words:")
print(" ".join(words[-100:]))

Sample filing length: 18357 words

First 200 words:
ITEM 1. BUSINESS ~~~~~~~ ~~~~~~~~ OVERVIEW ~~~~~~~~ Cummins Engine Company, Inc. ("Cummins" or "the Company") is a leading worldwide designer and manufacturer of fuel-efficient diesel engines and related products. Engines ranging from 76 to 2,000 horsepower serve a wide variety of equipment in Cummins' key markets: heavy-duty truck, midrange truck, power generation, bus and light commercial vehicles, industrial products, government and marine. In addition, Cummins produces strategic components and subsystems critical to the engine, including filters, turbochargers and electronic control systems. Cummins sells its products to original equipment manufacturers ("OEMs"), distributors and other customers worldwide and conducts manufacturing, sales, distribution and service activities in most areas of the world. In 1993, approximately 56 percent of net sales were made in the United States. Major international markets include the United King

### Data Cleaning
remove boilerplate at end (exhibits, signatures, leagal text)

In [ ]:
def clean_sec_filing(text):
  """
  Clean a SEC 10-K filing:
  1. Strip trailing exhibits/signatures/legal text
  2. Remove excessive whitespace
  3. Keep the substantive bussness/financial content
  """
  end_markers = [
      r"(?i)\bEXHIBIT\s+INDEX\b",
      r"(?i)\bSIGNATURES\b",
      r"(?i)\bPOWER\s+OF\s+ATTORNEY\b",
      r"(?i)\bEXHIBIT\s+\d",
  ]

  earliest_end = len(text)
  for pattern in end_markers:
    matches = list(re.finditer(pattern, text))
    if matches:
      last_match = matches[-1]
      if last_match.start() > 0.7 * len(text):
        earliest_end = min(earliest_end, last_match.start())

  text = text[:earliest_end]
  text = re.sub(r"\n{3,}", "\n\n", text)
  text = re.sub(r" {2,}", " ", text)
  text = text.strip()

  return text

# Clean all filings
clean_texts = []
for i, text in enumerate(all_texts):
  orignal_len = len(text.split())
  cleaned = clean_sec_filing(text)
  cleaned_len = len(cleaned.split())

  if cleaned_len > 500:
    clean_texts.append(cleaned)

  if i < 3:
    removed_pct = (1 - cleaned_len / orignal_len) * 100
    print(f"Filing {i}: {orignal_len} words -> {cleaned_len} words ({removed_pct:.1f}%)")

print(f"\n[OK] Cleaned {len(clean_texts)} filings.")

Filing 0: 18357 words -> 17521 words (4.6%)
Filing 1: 2727 words -> 2683 words (1.6%)
Filing 2: 6874 words -> 5451 words (20.7%)

[OK] Cleaned 100 filings.


### Chunking
divide sec filings to fit models context window

In [ ]:
def chunk_texts(texts, chunk_size=256, overlap=0.2):
  """
  Split texts into overlapping chunks.

  Args:
    texts: List of strings to chunk
    chunk_size: Number of tokens per chunk
    overlap: Overlap between chunks as a fraction
  """
  step = int(chunk_size * (1 - overlap))
  all_chunks = []

  for text in texts:
    words = text.split()
    for i in range(0, len(words), step):
      chunk = words[i:i + chunk_size]
      if len(chunk) > 50: # Skip tiny trailing chunks
        all_chunks.append(" ".join(chunk))

  return all_chunks


# Split into train and val First (at filing level, not chunk level)
# This prevent data leakage
random.seed(SEED)
random.shuffle(clean_texts)

train_filings = clean_texts[:NUM_TRAIN]
val_filings = clean_texts[NUM_TRAIN:NUM_TRAIN + NUM_VAL]

print(f"Train filings: {len(train_filings)}")
print(f"Val filings: {len(val_filings)}")

# Now chunk each set
train_chunks = chunk_texts(train_filings)
val_chunks = chunk_texts(val_filings)

print(f"Train chunks: {len(train_chunks)}")
print(f"Val chunks: {len(val_chunks)}")

print(f"\nSample fhunk (first 100 words)")
print(" ".join(train_chunks[0].split()[:100]))

Train filings: 80
Val filings: 20
Train chunks: 6830
Val chunks: 1191

Sample fhunk (first 100 words)
ITEM 1. BUSINESS. The following description of business is provided in accordance with General Instruction J.(2)(d). Associates First Capital Corporation ("First Capital" or the "Company"), a Delaware corporation, is an indirect subsidiary of Ford Motor Company ("Ford"). All the outstanding Common Stock of First Capital is owned by Ford Holdings, Inc. ("Holdings"). First Capital's principal operating subsidiary is Associates Corporation of North America ("Associates"), the second largest independent finance company in the United States as of September 30, 1993. Unless the context otherwise requires, reference to First Capital or Associates includes each parent company and all its subsidiaries. At December


In [ ]:
# Conver to HuggingFace Dataset format
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_chunks})
val_dataset = Dataset.from_dict({"text": val_chunks})

print(f"Train dataset: {(train_dataset)}")
print(f"Val dataset: {(val_dataset)}")

Train dataset: Dataset({
    features: ['text'],
    num_rows: 6830
})
Val dataset: Dataset({
    features: ['text'],
    num_rows: 1191
})


In [ ]:
# Measure base model preplexity on the ACTUAL val chunks
# We need this before we reload model for traning
val_sample = val_chunks[:50]
base_ppl_val = compute_preplixity(model, tokenizer, val_sample)
print(f"Base model preplexity on SEC validation chunks: {base_ppl_val:.1f}")
print("We'll compare this to the CPT model later")

Base model preplexity on SEC validation chunks: 19.1
We'll compare this to the CPT model later


### CPT with Unsloth + LoRA
1. Reload base model (fresh, no infrence mode)
2. Add LoRA adapters
3. Train on our SEC filing chunks

In [ ]:
# Free memory from the infrence model
del model
torch.cuda.empty_cache()

# Reload fresh for traning
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print(f"[OK] Fresh model loaded for training")


==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[OK] Fresh model loaded for training


In [ ]:
# Add LoRA adapters
#
# Key CPT Decisions:
# - rank=32: Higher than typical SFT (8-16) because we're teaching new knowledge, not just changing behaviour
# - target module includes embed_tokens + lm_head This is CPT-Specfic, SFT usually skips these. because we need the model to learn new domain vocablary
# - lora_alpha=32: Equal to rank, so effective scaling = 1

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention weights
        "gate_proj", "up_proj", "down_proj",      # FFN weights (where knowledge lives)
        "embed_tokens", "lm_head"                 # CPT-Specfic! (This is where model learns new vocab)
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False, # false means scalling factor is lora_alpha/r which means 32/32=1 so effectie scale is 1
)


# How many parameters are we actually trainning?
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nTotal parameters:     {total_params:>12,}")
print(f"Trainable (LoRA):     {trainable_params:>12,}  ({100*trainable_params/total_params:.2f}%)")
print(f"Frozen (base model):  {frozen_params:>12,}  ({100*frozen_params/total_params:.2f}%)")

Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Unsloth: Training embed_tokens in mixed precision to save VRAM

Total parameters:      149,414,208
Trainable (LoRA):       39,671,808  (26.55%)
Frozen (base model):   109,742,400  (73.45%)


In [ ]:
from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

# Training configuration -- every parameter explained:
training_args = UnslothTrainingArguments(
    output_dir="./cpt_sec_training_args",

    # --- Core traning ---
    num_train_epochs=2,               # Cycles through entire dataset
    per_device_train_batch_size=16,   # 16 chunks per GPU step
    gradient_accumulation_steps=2,     # num of chunks traning sees before updating weights (effective batch size = 16 * 2 = 32)

    # --- Learning Rate ---
    learning_rate=2e-4,               # Standard for LoRA
    embedding_learning_rate=2e-5,     # 10x Lower for embeddings!
                                      # critical to avoid catestopic forgetting
    warmup_steps=50,                  # Gentely increase amount of weight updates
    lr_scheduler_type="cosine",       # gentler than a step-decay and helps the model converge smoothly at the end of training

    # --- Optmization ---
    optim="adamw_8bit",               # 8-biy AdamW saves VRAM
    weight_decay=0.01,                # L2 regularization
    max_grad_norm=0.3,                # Gradient clipping to prevent spikes

    # --- Data handling ---
    max_length=MAX_SEQ_LENGTH,
    packing=True,                     # pack short texts together (no wasted padding!)
    dataset_text_field="text",

    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=16,

    # --- Check pointing ---
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- logging ---
    logging_strategy="steps",
    logging_steps=10,
    seed=SEED,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("[OK] Trainer configured. Ready to train!")
print(f"Training samples: {len(train_dataset)}")
print(f"Eval samples: {len(val_dataset)}")

Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/6830 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1191 [00:00<?, ? examples/s]

[OK] Trainer configured. Ready to train!
Training samples: 6830
Eval samples: 1191


In [ ]:
# TRAIN!
# On a T4 GPU with 135M model + 80 filings + 2 epochs, ~8-12 minutes
print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)
print("Watch the eval_loss -- that's your north star.")
print("It should decrease and then plateau.")
print()

trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

TRAINING STARTED
Watch the eval_loss -- that's your north star.
It should decrease and then plateau.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,830 | Num Epochs = 2 | Total steps = 428
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 39,671,808 of 202,498,368 (19.59% trained)


Unsloth: Setting lr = 2.00e-05 instead of 2.00e-04 for embed_tokens.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
25,2.740277,2.683481
50,2.602154,2.576529
75,2.529069,2.511080
100,2.427643,2.468775
125,2.382301,2.437526
150,2.388105,2.413412
175,2.355534,2.396563
200,2.379769,2.383623
225,2.342718,2.372346
250,2.288249,2.365539


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers fou

Step,Training Loss,Validation Loss
25,2.740277,2.683481
50,2.602154,2.576529
75,2.529069,2.511080
100,2.427643,2.468775
125,2.382301,2.437526
150,2.388105,2.413412
175,2.355534,2.396563
200,2.379769,2.383623
225,2.342718,2.372346
250,2.288249,2.365539


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")



TRAINING COMPLETE
Final train loss: 2.3812


### Comparision Before vs After

In [ ]:
# Switch to infrence mode
FastLanguageModel.for_inference(model)

# Generate from the same prompts we tried before
print("=" * 70)
print("AFTER CPT -- Financial Text Generation")
print("=" * 70)
for prompt in financial_prompts:
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
  print(f"\nPrompt: {prompt}")
  print(f"Output: {completion[:200]}")
  print("-" * 50)

Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER CPT -- Financial Text Generation


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: The company reported total revenue of
Output:  $15,074 in 1986 and net income for the years ended December 31, 1982. The Company also recorded a loss on its investment properties during January-December 1986 as an additional charge to this year's
--------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Risk factors include exposure to fluctuations in
Output:  the 50-state, nonunion payroll system. The Company has been unable or unwillingly forced by law enforcement agencies and other parties into compliance with these regulations as a result of its failur
--------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Diluted earnings per share for the fiscal year ended
Output:  December 31, 1987 were $.20/d and are presented in (Page omitted) Note to Consolidated Financial Statements at the end of each quarter except when one or more of these figures is less than half as mu
--------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: The Board of Directors declared a quarterly dividend of
Output:  10% on the Company's common stock. The dividends are paid in three periods each year beginning as follows: January, December and June (the "3-D" period): - January $25; - December $69 ($4); - June 7,
--------------------------------------------------

Prompt: Our long-term debt obligations consist primarily of
Output:  short term, current and prepayments. For the year ended December 31, 1992 $650 million was payable on behalf of investors; approximately $478 million is owed to creditors as a result of operations in
--------------------------------------------------


In [ ]:
# Preplexity comparision -- the real test
# Compare on the same val chunks we measured the base model on

cpt_ppl = compute_preplixity(model, tokenizer, val_sample)
print(f"\nPerplexity on SEC validation chunks:")
print(f"  Base model (before CPT):  {base_ppl_val:.1f}")
print(f"  After CPT:                {cpt_ppl:.1f}")
print(f"  Improvement:              {((base_ppl_val - cpt_ppl) / base_ppl_val * 100):.1f}%")
print()
if cpt_ppl < base_ppl_val:
  print("[PASS] Model is significantly less surprised by financial text!")
else:
  print("[WARN] Perplexity didn't improve -- might need more data or epochs")


Perplexity on SEC validation chunks:
  Base model (before CPT):  19.1
  After CPT:                13.2
  Improvement:              30.9%

[PASS] Model is significantly less surprised by financial text!


### The Forgetting Experiement

In [ ]:
# Test on general English text
general_prompts = [
    "The cat sat on the",
    "In a galaxy far far away",
    "The best way to learn programming is to",
    "Once upon a time there was a",
    "The weather today is expected to be",
]

print("=" * 70)
print("AFTER CPT -- General English (Forgetting Check)")
print("=" * 70)
for prompt in general_prompts:
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
  print(f"\nPrompt: {prompt}")
  print(f"Output: {completion[:200]}")
  print("-" * 50)

Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AFTER CPT -- General English (Forgetting Check)


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: The cat sat on the
Output:  chair. The committee of trustees voted to adopt an ordinance requiring that each resident in the United States, whether or not he is a UUA member (as defined by law), be provided with access rights f
--------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: In a galaxy far far away
Output: , the galaxy that we live in. We are living within our galaxy and it is not possible to tell what happens next if it was ever decided by us who will become its leader or vice versa." - Albert Einstein
--------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: The best way to learn programming is to
Output:  get hands-on experience. If you're interested in becoming a computer programmer, this means that most of the jobs available are open for those who have an undergraduate degree or higher and other qua
--------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt: Once upon a time there was a
Output:  great deal of controversy and suspicion as to whether or not the Company had an obligation under the Employee Stock Ownership Program (ESOP). The ESOP is generally accepted by employees, which means 
--------------------------------------------------

Prompt: The weather today is expected to be
Output:  more favorable than in the past. Weather conditions are predicted for a much warmer and drier atmosphere, with precipitation less frequent (52% of 1983 compared to 76%) over southern California as we
--------------------------------------------------


In [ ]:
# Perplexity on general text
general_texts = [
    "The weather was beautiful that morning as the children walked to school through the park.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The recipe calls for two cups of flour, one egg, and a tablespoon of butter.",
    "The basketball game went into overtime after a dramatic three-point shot.",
    "She opened the book and began reading the first chapter aloud to her students.",
]

general_ppl_after_cpt = compute_preplixity(model, tokenizer, general_texts)
print(f"Perplexity on general English (after CPT): {general_ppl_after_cpt:.1f}")

Perplexity on general English (after CPT): 32.0


- If this is much higher than expected, the model has 'forgotten'
- some general English. This is CATASTROPHIC FORGETTING in action.
- LoRA helps prevent this (base weights are frozen), but it's not perfect.
- Solution: mix in general data during training.

### CPT with Mixed Data(general english + SEC 10K filing)
80% domain data + 20% general data. This way model mantains its general capablites while learning domain knowledge

In [ ]:
general_dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
print(f"general_dataset: {general_dataset}")

# now we mix wiki data along with chunks
mixed_train_chunks = []
mixed_val_chunks = []

# --- Mix train chunks ---
for chunk in train_chunks:
    random_index = random.randrange(len(general_dataset))
    random_general_example = general_dataset[random_index]

    # Take 50 WORDS from the general example
    general_words = " ".join(random_general_example.get("text").split()[:50])

    mixed_train_chunks.append(chunk + " " + general_words)

# --- Mix val chunks ---
for chunk in val_chunks:
    random_index = random.randrange(len(general_dataset))
    random_general_example = general_dataset[random_index]

    # Take 50 WORDS from the general example
    general_words = " ".join(random_general_example.get("text").split()[:50])

    mixed_val_chunks.append(chunk + " " + general_words)

mixed_train_dataset = Dataset.from_dict({"text": mixed_train_chunks})
mixed_val_dataset = Dataset.from_dict({"text": mixed_val_chunks})

print(f"\nMixed_train_dataset: {mixed_train_dataset}")
print(f"\nMixed_val_dataset: {mixed_val_dataset}")

general_dataset: Dataset({
    features: ['text'],
    num_rows: 1801350
})

Mixed_train_dataset: Dataset({
    features: ['text'],
    num_rows: 6830
})

Mixed_val_dataset: Dataset({
    features: ['text'],
    num_rows: 1191
})


In [ ]:
# Reload model for traning
del model, trainer
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "k_proj", "q_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embed_tokens", "lm_head",
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False
)

print("[OK] Fresh model loaded for mixing experement")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Unsloth: Training embed_tokens in mixed precision to save VRAM
[OK] Fresh model loaded for mixing experement


In [ ]:
# Training configuration -- every parameter explained:
training_args_mix = UnslothTrainingArguments(
    output_dir="./cpt_sec_mixed_training_args",

    # --- Core traning ---
    num_train_epochs=2,               # Cycles through entire dataset
    per_device_train_batch_size=16,   # 16 chunks per GPU step
    gradient_accumulation_steps=2,    # num of chunks traning sees before updating weights (effective batch size = 16 * 2 = 32)

    # --- Learning Rate ---
    learning_rate=2e-4,               # Standard for LoRA
    embedding_learning_rate=2e-5,     # 10x Lower for embeddings!
                                      # critical to avoid catestopic forgetting
    warmup_steps=50,                  # Gentely increase amount of weight updates
    lr_scheduler_type="cosine",       # gentler than a step-decay and helps the model converge smoothly at the end of training

    # --- Optmization ---
    optim="adamw_8bit",               # 8-biy AdamW saves VRAM
    weight_decay=0.01,                # L2 regularization
    max_grad_norm=0.3,                # Gradient clipping to prevent spikes

    # --- Data handling ---
    max_length=MAX_SEQ_LENGTH,
    packing=True,                     # pack short texts together (no wasted padding!)
    dataset_text_field="text",

    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=16,

    # --- Check pointing ---
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- logging ---
    logging_strategy="steps",
    logging_steps=10,
    seed=SEED,
)

trainer_mix = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args_mix,
    train_dataset=mixed_train_dataset,
    eval_dataset=mixed_val_dataset,
)

print("[OK] Trainer configured. Ready to train!")
print(f"Mixed Training samples: {len(train_dataset)}")
print(f"Mixed Eval samples: {len(val_dataset)}")

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/6830 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=5):   0%|          | 0/6830 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/1191 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=5):   0%|          | 0/1191 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
[OK] Trainer configured. Ready to train!
Mixed Training samples: 6830
Mixed Eval samples: 1191


In [ ]:
print("=" * 70)
print("TRAINING WITH DATA MIXING (80/20)")
print("=" * 70)
trainer_mix_stats = trainer_mix.train()
print(f"\nFinal train loss: {trainer_mix_stats.training_loss:.4f}")

TRAINING WITH DATA MIXING (80/20)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,808 | Num Epochs = 2 | Total steps = 426
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 39,671,808 of 202,498,368 (19.59% trained)


Unsloth: Setting lr = 2.00e-05 instead of 2.00e-04 for embed_tokens.


Step,Training Loss,Validation Loss
25,2.670522,2.718442
50,2.699799,2.662597
75,2.584603,2.612240
100,2.563382,2.574949
125,2.544735,2.547400
150,2.503705,2.527074
175,2.472889,2.510252
200,2.463415,2.496762
225,2.477214,2.488889
250,2.355723,2.481418


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers fou


Final train loss: 2.4703


In [ ]:
# Compare: Mixed vs Pure domain training
FastLanguageModel.for_inference(model)

mixed_sec_ppl = compute_preplixity(model, tokenizer, val_sample)
mixed_general_ppl = compute_preplixity(model, tokenizer, general_texts)

print("=" * 70)
print("FORGETTING EXPERIMENT RESULTS")
print("=" * 70)
print()
print(f"{'Metric':<35} {'Base':>10} {'Pure CPT':>10} {'Mixed CPT':>10}")
print("-" * 70)
print(f"{'Perplexity (SEC filings)':.<35} {base_ppl_val:>10.1f} {cpt_ppl:>10.1f} {mixed_sec_ppl:>10.1f}")
print(f"{'Perplexity (General English)':.<35} {'--':>10} {general_ppl_after_cpt:>10.1f} {mixed_general_ppl:>10.1f}")

FORGETTING EXPERIMENT RESULTS

Metric                                    Base   Pure CPT  Mixed CPT
----------------------------------------------------------------------
Perplexity (SEC filings)...........       19.1       13.2       13.3
Perplexity (General English).......         --       32.0       28.0


**What to look for:**
- Pure CPT: great on SEC, worse on general (forgetting!)
- Mixed CPT: slightly worse on SEC, but maintains general ability
- This is the tradeoff. In production, you almost always want mixing.

### Can it Answer Finance questions?

In [ ]:
print("=" * 70)
print("THE LIMIT OF CPT")
print("=" * 70)
print()

question_prompts = [
    "Question: What was Apple's total revenue in fiscal year 2023?\nAnswer:",
    "Question: What is EBITDA and why do investors care about it?\nAnswer:",
    "Question: Explain the difference between operating income and net income.\nAnswer:",
]

for prompt in question_prompts:
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=80)
    print("-" * 50)
    print(f"Prompt: {prompt}")
    print(f"Output: {completion[:250]}")
    print("-" * 50)
    print()

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


THE LIMIT OF CPT



Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
Prompt: Question: What was Apple's total revenue in fiscal year 2023?
Answer:
Output:  Total revenues for the three years ended December 31, 2023 were $5.7 million (the following table includes changes from prior periods). The increase resulted primarily of lower sales and higher inventory levels due to a decrease in demand during thi
--------------------------------------------------



Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
Prompt: Question: What is EBITDA and why do investors care about it?
Answer:
Output:  The most commonly accepted measure of a company's cash flows (and therefore, its net income) from operations has been the Company's Earnings Per Share. This metric measures earnings after subtracting interest expense on common stock dividends which 
--------------------------------------------------

--------------------------------------------------
Prompt: Question: Explain the difference between operating income and net income.
Answer:
Output:  Operating Income = (Cost of Goods Sold) - (Operating Expenses) = Net Losses on Sale 60% Of Cost To Date Average Inventory 52% Total Assets at December 31, 1987 = (Net Sales from Period Ending in December) = (Total Days Since Year Ended ) -----------
--------------------------------------------------



**Observation:**
- Model keeps generating text instead of answering the question.
- CPT taught it financial LANGUAGE, not the financial BEHAVIOR.

To make it actually answer questions, we need Supervised Fine Tuning (SFT)!

### Save Model

In [ ]:
# Save LoRA adapter
SAVE_PATH = "./cpt_sec_adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH) if f.endswith(('.safetensors', '.bin'))
)

print(f"[OK] Adapter saved to {SAVE_PATH}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"Full model would be: ~270 MB")
print(f"Savings: {(1 - adapter_size / 270e6) * 100:.0f}%")

[OK] Adapter saved to ./cpt_sec_adapter
Adapter size: 215.4 MB
Full model would be: ~270 MB
Savings: 20%


---
## Summary

| Step | What | Why |
|------|------|-----|
| Base model test | Generated financial text with SmolLM | See it fail -- motivation for CPT |
| Data prep | Downloaded SEC 10-K filings, cleaned, chunked | Real-world data pipeline |
| CPT training | LoRA (rank 32) with Unsloth on domain text | Teach the model financial language |
| Before/after | Compared generation quality + perplexity | Prove it worked |
| Forgetting test | Checked general English after CPT | Understand the tradeoff |
| Data mixing | 80% domain + 20% general | Solution to catastrophic forgetting |
| SFT | Asked questions -- model can't answer | Next step after cpt |

**Key takeaway:** CPT teaches the model to SOUND like a domain. It does NOT teach the model to be HELPFUL. That's SFT.

**Next:** Planning to take this CPT'd model and fine-tune it on financial Q&A pairs (`virattt/financial-qa-10K`) so it actually answers questions about finance.